In [1]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
import pandas as pd

# Load mtcars dataset 
data = sm.datasets.get_rdataset("mtcars", "datasets")
df = data.data
print(f"Data shape: {df.shape}")
df['am_factor'] = df['am'].astype('category') # For binomial, often use 0/1 integers
df['am_numeric'] = df['am'] # Use 0 and 1 directly for the response

# --- 1. Logistic Regression (Binomial Family with Logit Link) ---
print("--- Logistic Regression (Binomial Family, Logit Link) ---")
# Define the GLM model
# The Binomial family has Logit as its default link function
logit_model = smf.glm(
    formula="am_numeric ~ hp + wt",
    data=df,
    family=sm.families.Binomial(link=sm.families.links.Logit())
).fit()

print("Family:", logit_model.family.__class__.__name__)
print("Link Function:", logit_model.family.link.__class__.__name__)

print("\nSummary for Logistic Model:")
print(logit_model.summary())

Data shape: (32, 11)
--- Logistic Regression (Binomial Family, Logit Link) ---
Family: Binomial
Link Function: Logit

Summary for Logistic Model:
                 Generalized Linear Model Regression Results                  
Dep. Variable:             am_numeric   No. Observations:                   32
Model:                            GLM   Df Residuals:                       29
Model Family:                Binomial   Df Model:                            2
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -5.0296
Date:                Sun, 28 Sep 2025   Deviance:                       10.059
Time:                        19:51:50   Pearson chi2:                     15.0
No. Iterations:                     8   Pseudo R-squ. (CS):             0.6453
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.0

In [2]:
df.sample(5)

,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb,am_factor,am_numeric
rownames,,,,,,,,,,,,,
Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4,1,1
Fiat X1-9,27.3,4,79.0,66,4.08,1.935,18.90,1,1,4,1,1,1
Honda Civic,30.4,4,75.7,52,4.93,1.615,18.52,1,1,4,2,1,1
Pontiac Firebird,19.2,8,400.0,175,3.08,3.845,17.05,0,0,3,2,0,0
Ford Pantera L,15.8,8,351.0,264,4.22,3.170,14.50,0,1,5,4,1,1


In [3]:
eq_params = logit_model.params
equation = f"log(p/(1-p)) = {eq_params['Intercept']:.4f} + {eq_params['hp']:.4f}*hp + {eq_params['wt']:.4f}*wt"
print(equation)

log(p/(1-p)) = 18.8663 + 0.0363*hp + -8.0835*wt


In [4]:
# Example: Predict probabilities
# Let's say a car has hp=100, wt=2.5
new_data_logit = pd.DataFrame({'hp': [100], 'wt': [2.5]})
linear_predictor_logit = logit_model.predict(new_data_logit, linear=True)[0]
# Use the inverse link function to get the probability
predicted_prob = logit_model.family.link.inverse(linear_predictor_logit)
print(f"\nPredicted linear predictor for hp=100, wt=2.5: {linear_predictor_logit:.4f}")
print(f"Predicted probability of manual transmission: {predicted_prob:.4f}")

# --- 2. Poisson Regression (Poisson Family with Log Link) ---
print("\n--- Poisson Regression (Poisson Family, Log Link) ---")
# For count data like 'cyl', which should be integers
poisson_model = smf.glm(
    formula="cyl ~ hp + mpg",
    data=df,
    family=sm.families.Poisson(link=sm.families.links.Log())
).fit()

print("Family:", poisson_model.family.__class__.__name__)
print("Link Function:", poisson_model.family.link.__class__.__name__)

print("\nSummary for Poisson Model:")
print(poisson_model.summary())

# Example: Predict counts
# Let's say a car has hp=100, mpg=25
new_data_poisson = pd.DataFrame({'hp': [100], 'mpg': [25]})
linear_predictor_poisson = poisson_model.predict(new_data_poisson, linear=True)[0]
# Use the inverse link function to get the expected count
predicted_count = poisson_model.family.link.inverse(linear_predictor_poisson)
print(f"\nPredicted linear predictor for hp=100, mpg=25: {linear_predictor_poisson:.4f}")
print(f"Predicted number of cylinders: {predicted_count:.4f}")


Predicted linear predictor for hp=100, wt=2.5: 2.2832
Predicted probability of manual transmission: 0.9075

--- Poisson Regression (Poisson Family, Log Link) ---
Family: Poisson
Link Function: Log

Summary for Poisson Model:
                 Generalized Linear Model Regression Results                  
Dep. Variable:                    cyl   No. Observations:                   32
Model:                            GLM   Df Residuals:                       29
Model Family:                 Poisson   Df Model:                            2
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -60.079
Date:                Sun, 28 Sep 2025   Deviance:                       3.4990
Time:                        19:51:50   Pearson chi2:                     3.46
No. Iterations:                     4   Pseudo R-squ. (CS):             0.3354
Covariance Type:            nonrobust                          

c:\Users\Han\anaconda3\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:985: FutureWarning: linear keyword is deprecated, use which="linear"
  warnings.warn(msg, FutureWarning)
c:\Users\Han\anaconda3\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:985: FutureWarning: linear keyword is deprecated, use which="linear"
  warnings.warn(msg, FutureWarning)
